# Test final: MAXIM S-2 vs shared residual MoE

Checkpoints fijados el **4 de octubre de 2026**, antes de consultar test:
- Baseline +10: mejor macro de validación, **época extra 10**.
- Shared head +10: mejor macro de validación, **época extra 8**, routing **oracle por tarea**.

Ejecutar en **Colab con GPU**. Configuración y rutas ya completas para `MyDrive/Facultad/tesis`.
Este notebook sólo hace inferencia, sin entrenar ni guardar checkpoints. Evalúa las cinco tareas sobre **4.730 pares**, un **crop central 256×256 por par**, RGB `[0,1]`, sin clipping, igual que validación. No equivale a evaluación de imagen completa ni al benchmark oficial del paper.

Guarda **un único JSON** con PSNR/SSIM por tarea, macro y ponderados, conteos y procedencia. No guarda imágenes, mapas SSIM ni resultados por imagen. Si se interrumpe, conserva tareas ya completadas; sólo repite la tarea incompleta. Un resultado terminado se muestra sin volver a evaluar. No cambiar modelos o hiperparámetros después de mirar test y seguir llamándolo test ciego.

El código de modelos se fija al commit del fork `Matiata/maxim` revisado; parámetros y shapes se verifican contra cada checkpoint. La instalación ocurre antes de importar JAX.


In [ ]:
# 1. Entorno del fork usado en los notebooks revisados
import os, sys, subprocess
from pathlib import Path
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
REPO_URL = 'https://github.com/Matiata/maxim.git'
REPO_COMMIT = '98984438176afb21694dd1ec29b02fd9b40426c1'
REPO_DIR = Path('/content/maxim_test_final_src')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(['git','-C',str(REPO_DIR),'remote','get-url','origin'], text=True).strip()
    if remote.rstrip('/') != REPO_URL:
        raise RuntimeError(f'Repositorio inesperado: {remote}')
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',REPO_COMMIT], check=True)
assert subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip() == REPO_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q',
                'jax[cuda12]==0.10.2','flax==0.11.2','ml-collections==0.1.1',
                'einops','Pillow'], check=True)
sys.path.insert(0, str(REPO_DIR))
print('Entorno instalado. Si JAX estaba importado antes, reiniciar el runtime y ejecutar desde el inicio.')


In [ ]:
# 2. Montar Drive e importar dependencias
from google.colab import drive
drive.mount('/content/gdrive')
import gc, hashlib, json, shutil, time, importlib.metadata
import numpy as np
import jax
import jax.numpy as jnp
from PIL import Image
from flax import traverse_util
from flax.core import unfreeze
from flax.training import checkpoints
from maxim.models.maxim import Model
from maxim.models.router import RouterModel
from maxim.models.moe import ResidualExpertHead, SharedResidualMaximMoE
from maxim.evaluation_metrics import aggregate_task_psnr, aggregate_task_ssim
if importlib.metadata.version('jax') != '0.10.2' or importlib.metadata.version('flax') != '0.11.2':
    raise RuntimeError('Versiones inesperadas; reiniciar runtime después de instalar.')
if not any(d.platform == 'gpu' for d in jax.devices()):
    raise RuntimeError('Seleccionar runtime GPU en Colab antes de esta evaluación.')
print('Dispositivos:', jax.devices())


In [ ]:
# 3. Protocolo fijado antes de la evaluación
TASKS = ['deblur','dehaze','denoise','derain','enhance']
DATA_ROOT = Path('/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier')
CHECKPOINT_ROOT = Path('/content/gdrive/MyDrive/Facultad/tesis/ckpts')
CACHE_ROOT = Path('/content/test_final_cache_20261004')
RESULT_PATH = CHECKPOINT_ROOT / 'test_final_baseline_vs_shared_20261004' / 'test_final_results.json'
PATCH_SIZE = 256
SSIM_DATA_RANGE, SSIM_FILTER_SIZE, SSIM_FILTER_SIGMA = 1.0, 11, 1.5
SSIM_K1, SSIM_K2 = 0.01, 0.03
MODEL_CONFIG = dict(variant='S-2', dropout_rate=0.1, num_outputs=3,
                    use_bias=True, num_supervision_scales=3)
MODEL_SPECS = [{'name': 'baseline_extra10', 'kind': 'baseline', 'relative_run': 'maxim_no_moe/baseline_S-2_from_newsplit_ckpt_extra10', 'epoch': 10, 'val_macro': 31.874760357289738, 'checkpoint_drive_url': 'https://drive.google.com/drive/folders/14AGPfW8aiXdYUClg4KtYVUy2P3dC_CnY'}, {'name': 'shared_extra10', 'kind': 'shared', 'relative_run': 'moe_shared_residual_from_newsplit_baseline_extra10', 'epoch': 8, 'val_macro': 32.06335479306456, 'checkpoint_drive_url': 'https://drive.google.com/drive/folders/1kiEeuMFzpPEvJZl1K9jlDjRP1V1ZgiK5'}]
EXPECTED_MANIFEST = {'deblur': {'train': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/train.txt', 'count': 7812, 'sha256': '9a2a5b0d07b3b5191c169f739bc1ac1c36b5504475cec49b487f422f206c8556'}, 'val': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/val.txt', 'count': 868, 'sha256': '2a0cae3d5174e3adf97b974e6618876c2fb9c1186e5b5b78794259af3267798c'}, 'test': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/test.txt', 'count': 3150, 'sha256': '1e10941c7d66a7d965f403ea1cb629973e136263a3055f646b269717a1782c00'}}, 'dehaze': {'train': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/train.txt', 'count': 561, 'sha256': '89bd2f54550404f9d3ab46871cf6a6479da9ccd6b6b94c42cee1549947ab7827'}, 'val': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/val.txt', 'count': 66, 'sha256': 'e2683e5bfffbc93b7851e13b6df11454de0d446ce83213c70129eca6a5eef8d4'}, 'test': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/test.txt', 'count': 383, 'sha256': '4ccb46026dfca10659bbac3a0e274c92e7fa11970b010072a479deec02a9873c'}}, 'denoise': {'train': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/train.txt', 'count': 3987, 'sha256': '8fc97cda0cfb50cb800b3b874836419710ab3a70f4acd991b205424d847e2733'}, 'val': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/val.txt', 'count': 463, 'sha256': '55c42c0d9f399903ce432e01d75eb00461d74278ffe4bc100d65183e85eaa0ec'}, 'test': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/test.txt', 'count': 767, 'sha256': '45fb0b36422f23d017ec2d95957043c2b64301e9423d16766f594cab618ed419'}}, 'derain': {'train': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/derain/train.txt', 'count': 3240, 'sha256': 'b121b7a2dc7f1ac2e4f854ebe837847a26ad57270d159db435a201a97901b8bd'}, 'val': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/derain/val.txt', 'count': 360, 'sha256': '38e94fbfe3508d7c373deb5ac0f7c59a18064c656baf045b1db59c3470d22557'}, 'test': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/derain/test.txt', 'count': 400, 'sha256': '5762ed7ecafe1dc00d87dc2d85dcb00be17369f6cfcafcc39d17b0df7e182012'}}, 'enhance': {'train': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/enhance/train.txt', 'count': 4923, 'sha256': '3c2e412e85538424e20d912525e36a7990abed179470fdd5db993f3cea0beeb7'}, 'val': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/enhance/val.txt', 'count': 546, 'sha256': '5ffdbe92a2f490449603279679bcb08c8b11eeea1bd3cb576b53e6337e24fb53'}, 'test': {'path': '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/enhance/test.txt', 'count': 30, 'sha256': '135c13a51a49b27b4320f8c96b55a0fcfcf4af63bb7fd2b0abea3dd9f3775464'}}}
SSIM_CONFIG = dict(data_range=1.0, filter_size=11, filter_sigma=1.5,
                   k1=0.01, k2=0.03, color_space='RGB', channel_reduction='mean',
                   padding='VALID', clipping='none')
EXPECTED_TEST_COUNTS = [EXPECTED_MANIFEST[t]['test']['count'] for t in TASKS]
assert EXPECTED_TEST_COUNTS == [3150,383,767,400,30]
assert sum(EXPECTED_TEST_COUNTS) == 4730
print('Checkpoints:', [(s['name'],s['epoch']) for s in MODEL_SPECS])
print('Test:', dict(zip(TASKS,EXPECTED_TEST_COUNTS)), '| total:', sum(EXPECTED_TEST_COUNTS))


In [ ]:
# 4. Identidad de listas y checkpoints: detener ante cualquier diferencia

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temp, path)

def validate_manifest(root, expected):
    actual, test_names = {}, {}
    for task in TASKS:
        actual[task] = {}
        for split in ('train','val','test'):
            path = Path(root) / task / f'{split}.txt'
            raw = path.read_bytes()
            names = [line.strip() for line in raw.decode('utf-8').splitlines() if line.strip()]
            value = {'count':len(names), 'sha256':hashlib.sha256(raw).hexdigest()}
            wanted = {k:expected[task][split][k] for k in ('count','sha256')}
            if value != wanted:
                raise ValueError(f'Lista modificada: {path}; {value} != {wanted}')
            if len(names) != len(set(names)):
                raise ValueError(f'Entradas duplicadas: {path}')
            for name in names:
                rel = Path(name)
                if rel.is_absolute() or '..' in rel.parts:
                    raise ValueError(f'Ruta inválida en {path}: {name}')
            actual[task][split] = value
            if split == 'test': test_names[task] = names
    return actual, test_names

ACTUAL_MANIFEST, TEST_NAMES = validate_manifest(DATA_ROOT, EXPECTED_MANIFEST)
for spec in MODEL_SPECS:
    run = CHECKPOINT_ROOT / spec['relative_run']
    selection = read_json(run / 'best_checkpoint' / 'best_metric.json')
    if selection['selection_metric'] != 'psnr_macro' or selection['epoch'] != spec['epoch']:
        raise ValueError(f'Cambió la selección de {spec["name"]}')
    if abs(selection['psnr_macro'] - spec['val_macro']) > 1e-6:
        raise ValueError(f'Cambió la métrica seleccionada de {spec["name"]}')
    if selection['task_names'] != TASKS or selection['task_count'] != [868,66,463,360,546]:
        raise ValueError('Orden de tareas o conteos de validación incompatibles')
    if selection['ssim_config'] != SSIM_CONFIG:
        raise ValueError('Configuración SSIM incompatible')
    metadata = read_json(run / 'run_metadata.json')
    for task in TASKS:
        for split in ('train','val','test'):
            for key in ('count','sha256'):
                if metadata['split_manifest'][task][split][key] != EXPECTED_MANIFEST[task][split][key]:
                    raise ValueError(f'La run {spec["name"]} usa otra lista')
    spec['checkpoint_path'] = str(run / 'best_checkpoint' / f'checkpoint_{spec["epoch"]}')
    if not Path(spec['checkpoint_path']).exists():
        raise FileNotFoundError(spec['checkpoint_path'])
    spec['selection'] = selection

PROTOCOL = {'split':'test', 'tasks':TASKS, 'split_manifest':ACTUAL_MANIFEST,
            'model_specs':MODEL_SPECS, 'model_config':MODEL_CONFIG,
            'repo_url':REPO_URL, 'repo_commit':REPO_COMMIT,
            'eval_batch_size':1, 'eval_patch_size':PATCH_SIZE,
            'preprocessing':'RGB float32 /255; aligned center crop; bottom/right reflect padding if small',
            'prediction':'last stage, last supervision scale; train=False; no clipping',
            'psnr':'mean per-image dB; MSE in 0..255 scale, floor=1e-6',
            'ssim_config':SSIM_CONFIG, 'shared_routing':'oracle',
            'extra_hidden_layers':2, 'correction_scale':1.0}
PROTOCOL_HASH = hashlib.sha256(json.dumps(PROTOCOL, sort_keys=True).encode()).hexdigest()
if RESULT_PATH.exists():
    REPORT = read_json(RESULT_PATH)
    if REPORT['protocol_sha256'] != PROTOCOL_HASH:
        raise RuntimeError('Existe un resultado de otro protocolo. Conservarlo y revisar la configuración.')
else:
    REPORT = {'status':'in_progress', 'protocol_sha256':PROTOCOL_HASH,
              'protocol':PROTOCOL, 'environment':{k:importlib.metadata.version(k)
                for k in ('jax','jaxlib','flax','orbax-checkpoint','numpy','Pillow')}, 'models':{}}
    atomic_json(RESULT_PATH, REPORT)
print('Protocolo verificado. Estado:', REPORT['status'])


Las listas train/val se leen sólo para comprobar su identidad; no se evalúan ni se usan para elegir modelos. La separación por escena se conserva de la auditoría previa; la igualdad de hashes no prueba por sí sola ausencia de fuga.


In [ ]:
# 5. Preparar copia local de test y validar decodificación, sin inferencia

def copy_verified_image(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(5):
        try:
            if not destination.exists():
                temp = destination.with_suffix(destination.suffix + '.tmp')
                shutil.copyfile(source, temp)
                os.replace(temp, destination)
            with Image.open(destination) as im:
                im.convert('RGB').load()
            return
        except (OSError, ValueError) as error:
            last_error = error
            destination.unlink(missing_ok=True)
            if attempt < 4: time.sleep(0.5)
    raise OSError(f'No se pudo copiar/decodificar {source}') from last_error

if REPORT['status'] != 'complete':
    for task in TASKS:
        for name in TEST_NAMES[task]:
            for folder in ('imgs','GT'):
                copy_verified_image(DATA_ROOT/task/folder/name, CACHE_ROOT/task/folder/name)
        print(f'Cache y decodificación OK: {task}, {len(TEST_NAMES[task])} pares')
else:
    print('Test ya terminado: se reutilizará el resultado guardado.')


In [ ]:
# 6. Funciones del baseline revisado, sin cambios de fórmula
def center_crop(image, target, crop_size):
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]
    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect",
        )
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect",
        )
    h = min(image.shape[0], target.shape[0])
    w = min(image.shape[1], target.shape[1])
    top = max(0, (h - crop_size) // 2)
    left = max(0, (w - crop_size) // 2)
    return (
        image[top:top + crop_size, left:left + crop_size],
        target[top:top + crop_size, left:left + crop_size],
    )

def compute_psnr_per_example(pred, target):
    diff = pred * 255.0 - target * 255.0
    mse = jnp.mean(diff ** 2, axis=(1, 2, 3))
    mse = jnp.maximum(mse, 1e-6)
    return 20.0 * jnp.log10(255.0 / jnp.sqrt(mse))

def compute_ssim_per_example(pred, target):
    """Single-scale RGB SSIM matching tf.image.ssim defaults."""
    pred = jnp.asarray(pred, dtype=jnp.float32)
    target = jnp.asarray(target, dtype=jnp.float32)
    if pred.ndim != 4 or pred.shape != target.shape:
        raise ValueError(f"SSIM expects equal NHWC tensors, got {pred.shape} and {target.shape}.")
    if pred.shape[1] < SSIM_FILTER_SIZE or pred.shape[2] < SSIM_FILTER_SIZE:
        raise ValueError(
            f"SSIM requires H,W >= {SSIM_FILTER_SIZE}, got {pred.shape[1:3]}."
        )
    coordinates = jnp.arange(SSIM_FILTER_SIZE, dtype=jnp.float32)
    coordinates -= (SSIM_FILTER_SIZE - 1.0) / 2.0
    gaussian_1d = jnp.exp(-0.5 * (coordinates / SSIM_FILTER_SIGMA) ** 2)
    gaussian_1d /= jnp.sum(gaussian_1d)
    gaussian_2d = gaussian_1d[:, None] * gaussian_1d[None, :]
    channels = pred.shape[-1]
    kernel = jnp.tile(gaussian_2d[:, :, None, None], (1, 1, 1, channels))

    def depthwise_filter(images):
        return jax.lax.conv_general_dilated(
            images, kernel, window_strides=(1, 1), padding="VALID",
            dimension_numbers=("NHWC", "HWIO", "NHWC"),
            feature_group_count=channels,
        )

    mean_pred = depthwise_filter(pred)
    mean_target = depthwise_filter(target)
    variance_pred = depthwise_filter(pred * pred) - mean_pred * mean_pred
    variance_target = depthwise_filter(target * target) - mean_target * mean_target
    covariance = depthwise_filter(pred * target) - mean_pred * mean_target
    c1 = (SSIM_K1 * SSIM_DATA_RANGE) ** 2
    c2 = (SSIM_K2 * SSIM_DATA_RANGE) ** 2
    luminance = (2.0 * mean_pred * mean_target + c1) / (
        mean_pred ** 2 + mean_target ** 2 + c1
    )
    contrast_structure = (2.0 * covariance + c2) / (
        variance_pred + variance_target + c2
    )
    return jnp.mean(luminance * contrast_structure, axis=(1, 2, 3))

# Chequeo sintético, sin consultar test.
probe = np.linspace(0,1,2*16*16*3,dtype=np.float32).reshape(2,16,16,3)
ssim_probe = np.asarray(jax.jit(compute_ssim_per_example)(jnp.asarray(probe),jnp.asarray(probe)))
np.testing.assert_allclose(ssim_probe,1.0,atol=1e-5)
assert np.isfinite(np.asarray(compute_psnr_per_example(jnp.asarray(probe),jnp.asarray(probe)))).all()
print('Métricas sintéticas OK')


In [ ]:
# 7. Restaurar únicamente variables del modelo; verificar cobertura completa

def make_model(kind):
    backbone = Model(**MODEL_CONFIG)
    if kind == 'baseline': return backbone
    if kind != 'shared': raise ValueError(kind)
    return SharedResidualMaximMoE(
        backbone, RouterModel(),
        [ResidualExpertHead(num_hidden_layers=2, zero_init_output=True) for _ in TASKS],
        routing_mode='oracle', correction_scale=1.0)

def validate_tree(expected, actual, label):
    wanted = traverse_util.flatten_dict(unfreeze(expected))
    found = traverse_util.flatten_dict(unfreeze(actual))
    missing, extra = set(wanted)-set(found), set(found)-set(wanted)
    if missing or extra:
        raise ValueError(f'{label}: parámetros incompatibles; faltan {list(missing)[:4]}, sobran {list(extra)[:4]}')
    for key in wanted:
        if wanted[key].shape != found[key].shape or np.dtype(wanted[key].dtype) != np.dtype(found[key].dtype):
            raise ValueError(f'{label}: shape/dtype incompatible en {key}')

def hash_variables(variables):
    digest = hashlib.sha256()
    for key, value in sorted(traverse_util.flatten_dict(unfreeze(variables)).items()):
        array = np.ascontiguousarray(jax.device_get(value))
        if not np.isfinite(array).all():
            raise FloatingPointError(f'Variables no finitas: {key}')
        digest.update(json.dumps(key).encode())
        digest.update(str(array.dtype).encode())
        digest.update(str(array.shape).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()

def load_model(spec):
    model = make_model(spec['kind'])
    dummy = jnp.zeros((1,PATCH_SIZE,PATCH_SIZE,3),dtype=jnp.float32)
    kwargs = {'task_id':jnp.zeros((1,),dtype=jnp.int32)} if spec['kind']=='shared' else {}
    template = jax.eval_shape(lambda: model.init(jax.random.PRNGKey(42),dummy,train=False,**kwargs))
    raw = checkpoints.restore_checkpoint(ckpt_dir=spec['checkpoint_path'], target=None)
    if not isinstance(raw,dict) or 'params' not in raw or 'step' not in raw:
        raise ValueError('No se restauró un TrainState completo; detener, sin fallback aleatorio.')
    expected_step = spec['epoch']*2000
    restored_step = int(np.asarray(raw['step']))
    if restored_step != expected_step:
        raise ValueError(f'Paso inesperado: {restored_step} != {expected_step}')
    variables = {'params':raw['params']}
    if raw.get('batch_stats') is not None:
        variables['batch_stats'] = raw['batch_stats']
    validate_tree(template,variables,spec['name'])
    fingerprint = hash_variables(variables)
    del raw, template
    gc.collect()
    variables = jax.tree_util.tree_map(jnp.asarray,variables)
    def step(variables,x,y,task_id):
        if spec['kind']=='shared':
            predictions,gates = model.apply(variables,x,train=False,task_id=task_id)
        else:
            predictions = model.apply(variables,x,train=False)
        final = predictions[-1][-1]
        if final.shape != y.shape:
            raise ValueError(f'Salida final incompatible: {final.shape} != {y.shape}')
        return compute_psnr_per_example(final,y), compute_ssim_per_example(final,y)
    # Variables como argumento dinámico; un JIT por modelo, batch/resolución fijos.
    return model, variables, jax.jit(step), fingerprint, restored_step

print('Restauración estricta preparada; no se creará ningún optimizador.')


In [ ]:
# 8. Evaluación final exhaustiva; reanudar por tareas sin seleccionar checkpoints

def read_pair(task, name):
    with Image.open(CACHE_ROOT/task/'imgs'/name) as im:
        x = np.asarray(im.convert('RGB'), dtype=np.float32)/255.0
    with Image.open(CACHE_ROOT/task/'GT'/name) as im:
        y = np.asarray(im.convert('RGB'), dtype=np.float32)/255.0
    x,y = center_crop(x,y,PATCH_SIZE)
    if x.shape != (PATCH_SIZE,PATCH_SIZE,3) or y.shape != x.shape:
        raise ValueError(f'Shapes incompatibles: {task}/{name}: {x.shape}, {y.shape}')
    return np.ascontiguousarray(x[None]),np.ascontiguousarray(y[None])

if REPORT['status'] != 'complete':
    for spec in MODEL_SPECS:
        record = REPORT['models'].get(spec['name'],{})
        model,variables,step,fingerprint,restored_step = load_model(spec)
        if record and record['variables_sha256'] != fingerprint:
            raise RuntimeError('Cambió el checkpoint durante una reanudación; no mezclar resultados.')
        if not record:
            record = {'variables_sha256':fingerprint,'restored_step':restored_step,
                      'checkpoint_path':spec['checkpoint_path'], 'selected_epoch':spec['epoch'],
                      'selected_by':'psnr_macro_validation','tasks':{}}
            REPORT['models'][spec['name']] = record
            atomic_json(RESULT_PATH,REPORT)
        for task_id,task in enumerate(TASKS):
            if task in record['tasks']:
                if record['tasks'][task]['count'] != EXPECTED_TEST_COUNTS[task_id]:
                    raise ValueError('Conteo guardado inválido')
                print(f'{spec["name"]} / {task}: resultado completo reutilizado')
                continue
            psnr_sum,ssim_sum,count = 0.0,0.0,0
            for name in TEST_NAMES[task]:
                x,y = read_pair(task,name)
                p,s = step(variables,jnp.asarray(x),jnp.asarray(y),jnp.asarray([task_id],dtype=jnp.int32))
                p,s = float(np.asarray(jax.device_get(p))[0]),float(np.asarray(jax.device_get(s))[0])
                if not np.isfinite([p,s]).all():
                    raise FloatingPointError(f'Métrica no finita: {task}/{name}')
                psnr_sum += p; ssim_sum += s; count += 1
            if count != EXPECTED_TEST_COUNTS[task_id]:
                raise ValueError(f'Test incompleto: {task}, {count}')
            record['tasks'][task] = {'count':count,'psnr':psnr_sum/count,'ssim':ssim_sum/count}
            atomic_json(RESULT_PATH,REPORT)
            print(f'{spec["name"]} / {task}: n={count}, PSNR={psnr_sum/count:.4f} dB, SSIM={ssim_sum/count:.4f}')
        counts = [record['tasks'][t]['count'] for t in TASKS]
        p = aggregate_task_psnr([record['tasks'][t]['psnr']*record['tasks'][t]['count'] for t in TASKS],counts)
        s = aggregate_task_ssim([record['tasks'][t]['ssim']*record['tasks'][t]['count'] for t in TASKS],counts)
        record['summary'] = {k:float(p[k]) for k in ('psnr_macro','psnr_weighted')}
        record['summary'].update({k:float(s[k]) for k in ('ssim_macro','ssim_weighted')})
        record['summary']['sample_count'] = sum(counts)
        atomic_json(RESULT_PATH,REPORT)
        del step,variables,model
        jax.clear_caches(); gc.collect()
    REPORT['status'] = 'complete'
    atomic_json(RESULT_PATH,REPORT)
print('Estado final:',REPORT['status'])


In [ ]:
# 9. Tabla final y diferencia shared − baseline; sin cambiar la selección
from IPython.display import display, Markdown
if REPORT['status'] != 'complete':
    raise RuntimeError('Evaluación incompleta; ejecutar la celda anterior.')
b = REPORT['models']['baseline_extra10']
s = REPORT['models']['shared_extra10']
rows = ['| Tarea | n | PSNR baseline | PSNR shared | Δ dB | SSIM baseline | SSIM shared | Δ SSIM |',
        '|---|---:|---:|---:|---:|---:|---:|---:|']
for task in TASKS:
    bv,sv = b['tasks'][task],s['tasks'][task]
    rows.append(f'| {task} | {bv["count"]} | {bv["psnr"]:.4f} | {sv["psnr"]:.4f} | {sv["psnr"]-bv["psnr"]:+.4f} | {bv["ssim"]:.4f} | {sv["ssim"]:.4f} | {sv["ssim"]-bv["ssim"]:+.4f} |')
for label,suffix in [('Macro','macro'),('Ponderado','weighted')]:
    bp,sp = b['summary'][f'psnr_{suffix}'],s['summary'][f'psnr_{suffix}']
    bs,ss = b['summary'][f'ssim_{suffix}'],s['summary'][f'ssim_{suffix}']
    rows.append(f'| **{label}** | 4730 | {bp:.4f} | {sp:.4f} | {sp-bp:+.4f} | {bs:.4f} | {ss:.4f} | {ss-bs:+.4f} |')
display(Markdown('\n'.join(rows)))
print('Resultado:',RESULT_PATH)
print('Interpretación: una seed, routing oracle, crops centrales. El test no mide variabilidad entre entrenamientos ni costo de inferencia.')


**Cómo interpretar:** reportar la diferencia observada y los cambios por tarea, incluso si contradicen validación. Si la ventaja se mantiene, la conclusión es sobre esta variante shared residual/oracle en este protocolo. Una sola seed no demuestra robustez entre entrenamientos; tampoco separa MoE de mayor cantidad de parámetros. Para confirmar esa atribución haría falta una ablación con capacidad comparable. `Enhance` tiene sólo 30 pares de test. No agregar otras métricas ni utilizar test para elegir nuevas épocas.
